# SDTM Findings — Diff vs pre-2026-03 baseline

Compares freshly regenerated `Specimen_Findings.xlsx` and `Measurement_Findings.xlsx`
against the snapshots under `machine_actionable/pre-2026-03/`.

Both files have two diffable sheets:
- **Test_Identity** — one row per NCIt-coded test, key = `NCIt_Code`
- **Measurement_Specs** — one row per DSS (or test without DSS), key = `(TESTCD, DS_Code)`

Outputs added / removed / changed at row level for each sheet of each artifact.

Note: Instrument_Findings.xlsx is *not* included (out of scope for the lab-test path).
Run a separate diff for it if needed.


In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..') / 'machine_actionable'
BASELINE = ROOT / 'pre-2026-03'

ARTIFACTS = ['Specimen_Findings.xlsx', 'Measurement_Findings.xlsx']


In [ ]:
def diff_keyed(old, new, key_cols):
    """Return (added, removed, changed) for two dfs with the given key."""
    if isinstance(key_cols, str):
        key_cols = [key_cols]
    o = old.copy(); n = new.copy()
    for k in key_cols:
        o[k] = o[k].fillna('')
        n[k] = n[k].fillna('')
    o['_key'] = o[key_cols].astype(str).agg('||'.join, axis=1)
    n['_key'] = n[key_cols].astype(str).agg('||'.join, axis=1)
    o = o.dropna(subset=key_cols, how='all')
    n = n.dropna(subset=key_cols, how='all')
    old_keys = set(o['_key'])
    new_keys = set(n['_key'])
    added = n[n['_key'].isin(new_keys - old_keys)].drop(columns='_key').copy()
    removed = o[o['_key'].isin(old_keys - new_keys)].drop(columns='_key').copy()
    common = sorted(old_keys & new_keys)
    oc = o[o['_key'].isin(common)].drop(columns='_key').set_index(key_cols).sort_index()
    nc = n[n['_key'].isin(common)].drop(columns='_key').set_index(key_cols).sort_index()
    cols = [c for c in oc.columns if c in nc.columns]
    oc = oc[cols]; nc = nc[cols]
    o_str = oc.fillna('').astype(str)
    n_str = nc.fillna('').astype(str)
    diff_mask = (o_str != n_str)
    rows = []
    for idx in oc.index[diff_mask.any(axis=1)]:
        for c in cols:
            if o_str.at[idx, c] != n_str.at[idx, c]:
                rec = {k: (idx[i] if isinstance(idx, tuple) else idx) for i, k in enumerate(key_cols)}
                rec['column'] = c
                rec['old'] = oc.at[idx, c]
                rec['new'] = nc.at[idx, c]
                rows.append(rec)
    changed = pd.DataFrame(rows)
    return added, removed, changed


In [ ]:
results = {}
for art in ARTIFACTS:
    print('=' * 60)
    print(art)
    print('=' * 60)
    art_results = {}
    for sheet, key in [('Test_Identity', 'NCIt_Code'), ('Measurement_Specs', ['TESTCD','DS_Code'])]:
        old = pd.read_excel(BASELINE / art, sheet_name=sheet)
        new = pd.read_excel(ROOT / art, sheet_name=sheet)
        added, removed, changed = diff_keyed(old, new, key)
        art_results[sheet] = (old, new, added, removed, changed)
        print(f'  [{sheet}]  baseline={len(old)}  new={len(new)}  added={len(added)}  removed={len(removed)}  changed_cells={len(changed)}')
        if len(changed):
            top = changed['column'].value_counts().head(5).to_dict()
            print(f'    top changed columns: {top}')
    results[art] = art_results
    print()


## Specimen_Findings — Test_Identity inspection


In [ ]:
_, _, added, removed, changed = results['Specimen_Findings.xlsx']['Test_Identity']
added.head(20)


In [ ]:
removed.head(20)


In [ ]:
changed.head(40)


## Specimen_Findings — Measurement_Specs inspection


In [ ]:
_, _, added, removed, changed = results['Specimen_Findings.xlsx']['Measurement_Specs']
added.head(20)


In [ ]:
removed.head(20)


In [ ]:
changed.head(40)


## Measurement_Findings — Test_Identity inspection


In [ ]:
_, _, added, removed, changed = results['Measurement_Findings.xlsx']['Test_Identity']
added.head(20)


In [ ]:
removed.head(20)


In [ ]:
changed.head(40)


## Measurement_Findings — Measurement_Specs inspection


In [ ]:
_, _, added, removed, changed = results['Measurement_Findings.xlsx']['Measurement_Specs']
added.head(20)


In [ ]:
removed.head(20)


In [ ]:
changed.head(40)


## Optional: write diff CSVs

Uncomment to persist diff outputs alongside the SDTM CT and COSMoS diffs.


In [ ]:
OUT = ROOT / 'diffs' / 'pre-2026-03__current'
OUT.mkdir(parents=True, exist_ok=True)
for art, sheets in results.items():
    stem = Path(art).stem
    for sheet, (_, _, added, removed, changed) in sheets.items():
        added.to_csv(OUT / f'{stem}__{sheet}__added.csv', index=False)
        removed.to_csv(OUT / f'{stem}__{sheet}__removed.csv', index=False)
        changed.to_csv(OUT / f'{stem}__{sheet}__changed.csv', index=False)
print('wrote', OUT)
